# Experiment 13 — Two-layer AutoGluon stack

Layer 1 is the Exp 9 winning AutoGluon ensemble on its best 18-feature set. Layer 2 receives the same 18 features plus `layer1_prediction` and learns a second ensemble.

Crucially, Layer 2 never trains on Layer 1 in-sample predictions. Its training prediction is chronological out-of-fold (OOF): each prediction is made by a Layer 1 model fit only on earlier months. The outer validation period is not used in either layer's fitting or tuning.

In [1]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
from autogluon.tabular import TabularPredictor
from scipy.spatial import cKDTree
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
roots = ([Path(configured_root)] if configured_root else []) + [Path.cwd(), *Path.cwd().parents]
project_root = next((root for root in roots if (root / 'exploration/metul/preprocessing.py').exists()), None)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
experiment_dir = project_root / 'exploration' / 'metul'
sys.path.insert(0, str(experiment_dir))
from preprocessing import RentalPreprocessor, load_datasets

TARGET = 'MONTHLY_RENT'
GROUP_COLUMNS = ['TOWN', 'FLAT_TYPE']
LAG_MONTHS = 24
N_OOF_SPLITS = 3
LAYER1_TIME_LIMIT = int(os.environ.get('STACK_LAYER1_TIME_LIMIT', '180'))
LAYER2_TIME_LIMIT = int(os.environ.get('STACK_LAYER2_TIME_LIMIT', '300'))
RUN_ID = os.environ.get('STACK_RUN_ID', pd.Timestamp.now().strftime('%Y%m%d_%H%M%S'))
MODEL_HYPERPARAMETERS = {'GBM': {}, 'CAT': {}}
PRESETS = ['medium_quality', 'ignore_text']

compact_features = [
    'TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', 'FLOOR_AREA_SQM',
    'approval_date_num', 'approval_month', 'coe_quota_premium_mean',
    'YEAR_COMPLETED', 'MAX_FLOOR', 'mrt_dist_nearest_km', 'mall_dist_nearest_km',
]
# This was the winning Exp 9 variant: validation RMSE 467.122.
BEST_FEATURES = compact_features + [
    'monthly_target_lag', 'town_flat_type_target_mean', 'nearest_mrt_name',
    'nearest_tourist_attraction_name', 'nearest_monument_name',
]


In [2]:
class TargetFeatureTransformer:
    """Target features fit only on the supplied labelled history."""

    def fit(self, history):
        self.global_mean_ = history[TARGET].mean()
        self.group_mean_ = history.groupby(GROUP_COLUMNS)[TARGET].mean()
        self.monthly_mean_ = history.groupby('period')[TARGET].mean()
        self.group_period_ = history.groupby(GROUP_COLUMNS + ['period'], as_index=False)[TARGET].mean()
        self.lag_lookup_ = self.group_period_.rename(columns={'period': 'source_period', TARGET: 'town_flat_type_target_lag'})
        return self

    def transform(self, frame):
        result = frame[GROUP_COLUMNS + ['period']].copy()
        result['monthly_target_lag'] = (result['period'] - LAG_MONTHS).map(self.monthly_mean_).fillna(self.global_mean_)
        result['source_period'] = result['period'] - LAG_MONTHS
        result = result.merge(self.lag_lookup_, on=GROUP_COLUMNS + ['source_period'], how='left').drop(columns='source_period')
        group_mean = pd.Series(result.set_index(GROUP_COLUMNS).index.map(self.group_mean_), index=result.index)
        result['town_flat_type_target_mean'] = group_mean.fillna(self.global_mean_)
        result['town_flat_type_target_lag'] = result['town_flat_type_target_lag'].fillna(result['town_flat_type_target_mean'])
        return result[['monthly_target_lag', 'town_flat_type_target_lag', 'town_flat_type_target_mean']]

    def fit_transform_train(self, history):
        self.fit(history)
        monthly = history.groupby(GROUP_COLUMNS + ['period'], as_index=False)[TARGET].agg(['sum', 'count']).reset_index()
        monthly = monthly.sort_values(GROUP_COLUMNS + ['period'])
        monthly['prior_sum'] = monthly.groupby(GROUP_COLUMNS)['sum'].cumsum() - monthly['sum']
        monthly['prior_count'] = monthly.groupby(GROUP_COLUMNS)['count'].cumsum() - monthly['count']
        monthly['town_flat_type_target_mean'] = monthly['prior_sum'] / monthly['prior_count'].replace(0, np.nan)
        global_monthly = history.groupby('period')[TARGET].agg(['sum', 'count']).sort_index()
        global_prior = (global_monthly['sum'].cumsum() - global_monthly['sum']) / (global_monthly['count'].cumsum() - global_monthly['count']).replace(0, np.nan)
        result = history[GROUP_COLUMNS + ['period']].merge(monthly[GROUP_COLUMNS + ['period', 'town_flat_type_target_mean']], on=GROUP_COLUMNS + ['period'], how='left')
        result['town_flat_type_target_mean'] = result['town_flat_type_target_mean'].fillna(result['period'].map(global_prior))
        result['monthly_target_lag'] = (result['period'] - LAG_MONTHS).map(self.monthly_mean_).fillna(result['period'].map(global_prior))
        return result[['monthly_target_lag', 'town_flat_type_target_mean']]

def nearest_mrt_name(base):
    stations = datasets['mrt'][['NAME', 'LATITUDE', 'LONGITUDE']].dropna()
    _, indices = cKDTree(stations[['LATITUDE', 'LONGITUDE']].to_numpy() * [111.0, 110.96]).query(base[['LATITUDE', 'LONGITUDE']].to_numpy() * [111.0, 110.96])
    return pd.Series(stations['NAME'].astype(object).to_numpy()[indices], name='nearest_mrt_name')

def nearest_geojson_name(base, filename, field, feature_name):
    with (project_root / 'data' / 'possible_additional_features' / filename).open() as handle:
        geojson = json.load(handle)
    points = pd.DataFrame([
        {'longitude': feature['geometry']['coordinates'][0], 'latitude': feature['geometry']['coordinates'][1], feature_name: feature['properties'][field]}
        for feature in geojson['features'] if feature.get('geometry', {}).get('type') == 'Point'
    ]).dropna()
    _, indices = cKDTree(points[['longitude', 'latitude']].to_numpy() * [110.96, 111.0]).query(base[['LONGITUDE', 'LATITUDE']].to_numpy() * [110.96, 111.0])
    return pd.Series(points[feature_name].astype(object).to_numpy()[indices], name=feature_name)

def model_features(history_raw, future_raw):
    """Build Exp 9's winning features, fitting all learned steps on history_raw only."""
    preprocessor = RentalPreprocessor(datasets)
    history_base = preprocessor.fit_transform(history_raw.drop(columns=['period', TARGET], errors='ignore'))
    future_base = preprocessor.transform(future_raw.drop(columns=['period', TARGET], errors='ignore'))
    history_labels = history_base[GROUP_COLUMNS].reset_index(drop=True).assign(period=history_raw['period'].to_numpy(), **{TARGET: history_raw[TARGET].to_numpy()})
    future_labels = future_base[GROUP_COLUMNS].reset_index(drop=True).assign(period=future_raw['period'].to_numpy())
    transformer = TargetFeatureTransformer()
    history_target = transformer.fit_transform_train(history_labels)
    future_target = transformer.transform(future_labels)
    def combine(base, target):
        return pd.concat([
            base[compact_features].reset_index(drop=True), target.reset_index(drop=True),
            nearest_mrt_name(base).reset_index(drop=True),
            nearest_geojson_name(base, 'TouristAttractions.geojson', 'PAGETITLE', 'nearest_tourist_attraction_name').reset_index(drop=True),
            nearest_geojson_name(base, 'MonumentsGEOJSON.geojson', 'NAME', 'nearest_monument_name').reset_index(drop=True),
        ], axis=1)[BEST_FEATURES]
    return combine(history_base, history_target), combine(future_base, future_target)


In [3]:
_, datasets = load_datasets(project_root / 'data')
raw = datasets['train'].copy()
raw['period'] = pd.to_datetime(raw['RENT_APPROVAL_DATE'] + '-01').dt.to_period('M')
periods = np.sort(raw['period'].unique())
cutoff = periods[int(len(periods) * 0.80)]
raw_fit = raw.loc[raw['period'] < cutoff].copy()
raw_valid = raw.loc[raw['period'] >= cutoff].copy()
print(f'Outer cutoff={cutoff}; fit={len(raw_fit):,}; validation={len(raw_valid):,}')

def train_predictor(train_x, train_y, path, time_limit, tuning_data=None):
    predictor = TabularPredictor(label=TARGET, problem_type='regression', eval_metric='root_mean_squared_error', path=str(path))
    fit_args = dict(train_data=train_x.assign(**{TARGET: train_y.to_numpy()}), hyperparameters=MODEL_HYPERPARAMETERS, presets=PRESETS, time_limit=time_limit, verbosity=2)
    if tuning_data is not None:
        fit_args['tuning_data'] = tuning_data
    return predictor.fit(**fit_args)

def layer1_oof(raw_history, run_name):
    history_periods = np.sort(raw_history['period'].unique())
    parts = []
    for fold, (early_index, later_index) in enumerate(TimeSeriesSplit(n_splits=N_OOF_SPLITS).split(history_periods), start=1):
        early_raw = raw_history.loc[raw_history['period'].isin(history_periods[early_index])].copy()
        later_raw = raw_history.loc[raw_history['period'].isin(history_periods[later_index])].copy()
        early_x, later_x = model_features(early_raw, later_raw)
        predictor = train_predictor(early_x, early_raw[TARGET].astype(float), experiment_dir / 'ag_exp_13_models' / RUN_ID / run_name / f'layer1_oof_fold_{fold}', LAYER1_TIME_LIMIT)
        part = later_x.copy()
        part['layer1_prediction'] = predictor.predict(later_x).astype(float).clip(lower=0).to_numpy()
        part[TARGET] = later_raw[TARGET].to_numpy()
        part['fold'] = fold
        parts.append(part)
        print(f'Layer 1 OOF fold {fold}: fit={len(early_raw):,}; OOF={len(later_raw):,}')
    return pd.concat(parts, ignore_index=True)

oof = layer1_oof(raw_fit, 'validation')
assert oof['layer1_prediction'].notna().all()
oof['layer1_residual'] = oof[TARGET] - oof['layer1_prediction']
display(oof[['layer1_prediction', TARGET, 'layer1_residual']].describe())


Outer cutoff=2024-05; fit=117,988; validation=32,012
[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and re

Verbosity: 2 (Standard Logging)


[categorical_clean] derived HDB feature | REGION: 5 -> 5 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.82 GB / 16.00 GB (30.1%)
Disk Space Avail:   170.31 GB / 460.38 GB (37.0%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 180s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_13_models/20261012_103855/validation/layer1_oof_fold_1"
Train Data Rows:    31565
Train Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generators to preprocess the data .

Layer 1 OOF fold 1: fit=31,565; OOF=28,514
[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyph

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.68 GB / 16.00 GB (29.2%)
Disk Space Avail:   170.30 GB / 460.38 GB (37.0%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 180s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_13_models/20261012_103855/validation/layer1_oof_fold_2"
Train Data Rows:    60079
Train Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Gen

Layer 1 OOF fold 2: fit=60,079; OOF=27,858
[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyph

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.76 GB / 16.00 GB (29.7%)
Disk Space Avail:   170.29 GB / 460.38 GB (37.0%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 180s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_13_models/20261012_103855/validation/layer1_oof_fold_3"
Train Data Rows:    87937
Train Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Gen

Layer 1 OOF fold 3: fit=87,937; OOF=30,051


,layer1_prediction,MONTHLY_RENT,layer1_residual
count,86423.000000,86423.000000,86423.000000
mean,2574.190545,2791.240549,217.050004
std,515.285128,743.320096,572.729411
min,1289.968140,300.000000,-3873.494629
25%,2176.152344,2200.000000,-137.619995
50%,2512.872559,2800.000000,201.413574
75%,2912.936890,3300.000000,584.700317
max,5138.457031,7600.000000,3838.455078


In [4]:
# The final OOF fold is reserved to choose Layer 2's ensemble; it is still part
# of outer-fit history, never part of the outer validation period.
stop_fold = oof['fold'].max()
layer2_train = oof.loc[oof['fold'] < stop_fold].drop(columns=['fold', 'layer1_residual']).copy()
layer2_stop = oof.loc[oof['fold'] == stop_fold].drop(columns=['fold', 'layer1_residual']).copy()
layer2_predictor = train_predictor(
    layer2_train.drop(columns=TARGET), layer2_train[TARGET].astype(float),
    experiment_dir / 'ag_exp_13_models' / RUN_ID / 'validation' / 'layer2',
    LAYER2_TIME_LIMIT, tuning_data=layer2_stop,
)

# Fit Layer 1 on all outer-fit rows. No outer-validation labels are supplied to it.
fit_x, valid_x = model_features(raw_fit, raw_valid)
layer1_predictor = train_predictor(
    fit_x, raw_fit[TARGET].astype(float),
    experiment_dir / 'ag_exp_13_models' / RUN_ID / 'validation' / 'layer1_full',
    LAYER1_TIME_LIMIT,
)
layer1_valid_prediction = layer1_predictor.predict(valid_x).astype(float).clip(lower=0)
layer2_valid_x = valid_x.copy()
layer2_valid_x['layer1_prediction'] = layer1_valid_prediction.to_numpy()
layer2_valid_prediction = layer2_predictor.predict(layer2_valid_x).astype(float).clip(lower=0)

y_valid = raw_valid[TARGET].astype(float).reset_index(drop=True)
def metrics(name, prediction):
    return {'model': name, 'RMSE': mean_squared_error(y_valid, prediction) ** 0.5, 'MAE': mean_absolute_error(y_valid, prediction), 'R2': r2_score(y_valid, prediction)}

comparison = pd.DataFrame([
    metrics('Layer 1: Exp 9 feature ensemble', layer1_valid_prediction),
    metrics('Layer 2: feature ensemble + Layer 1 prediction', layer2_valid_prediction),
])
display(comparison.sort_values('RMSE'))

# The second row must beat Layer 1 on this untouched period to justify stacking.


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.73 GB / 16.00 GB (29.6%)
Disk Space Avail:   170.24 GB / 460.38 GB (37.0%)
Presets specified: ['medium_quality', 'ignore_text']
	You should ensure the dtypes match to avoid bugs or instability.
	AutoGluon will attempt to convert the dtypes to align.
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_13_models/20261012_103855/validation/layer2"
Train Data Rows:    56372
Train Data Colu

[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.60 GB / 16.00 GB (28.7%)
Disk Space Avail:   170.23 GB / 460.38 GB (37.0%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 180s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_13_models/20261012_103855/validation/layer1_full"
Train Data Rows:    117988
Train Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generato

,model,RMSE,MAE,R2
0,Layer 1: Exp 9 feature ensemble,468.271759,340.986810,0.491986
1,Layer 2: feature ensemble + Layer 1 prediction,538.150600,387.430734,0.329055
